> 📘 **Solucionario.** Esta versión contiene las soluciones de todos los ejercicios; está pensada para el equipo docente.

<a href="https://colab.research.google.com/github/ibonfilrivera/Curso_Python_FQ/blob/main/soluciones/Sesion_3_soluciones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# **Curso introductorio de Python**
## **Facultad de Química, UNAM** · Departamento de Física y Química Teórica

**Elaboraron:** Iván Bonfil, Rafael Rodriguez, Roberto Rojas y Lizeth Franco Nolasco

---

# **Sesión 3: Bibliotecas científicas y visualización de datos**


En esta sesión usaremos bibliotecas especializadas de Python para ajustar datos experimentales,
generar gráficas de calidad editorial, analizar las lecturas de un espectrofotómetro y explorar
una base de datos con casi 10 000 compuestos.

**Al terminar podrás:**
- Hacer regresiones y resolver ecuaciones con SciPy.
- Construir gráficas científicas con Matplotlib (ejes, unidades, leyendas y exportación).
- Leer, filtrar, agrupar y resumir datos de laboratorio con Pandas.
- Determinar experimentalmente la ley de Lambert-Beer y su intervalo de validez.
- Representar moléculas y buscar subestructuras con RDKit.

## **¿Cómo usar este notebook?**

Este notebook es **interactivo**: cada ejercicio tiene una celda de código para tu respuesta y
una celda que la **verifica automáticamente**, como en los cursos de [Kaggle Learn](https://www.kaggle.com/learn).

1. Ejecuta la celda de **configuración** que está justo abajo (una sola vez, al abrir el notebook).
2. En cada ejercicio, sustituye los espacios `____` por tu código y ejecuta la celda.
3. Ejecuta la celda `ejN.verificar()`. Verás uno de estos mensajes:
   - ✅ **¡Correcto!** — puedes continuar.
   - ❌ **Incorrecto** — el mensaje te dice qué revisar.
   - ✏️ **Pendiente** — aún falta completar algo.
4. Si te atoras, quita el `#` de `ejN.pista()` para recibir una pista, o de `ejN.solucion()`
   para ver una solución. **Intenta resolverlo antes de ver la solución.**
5. Ejecuta `progreso()` en cualquier momento para ver tu avance en la sesión.

> 📋 **Registro de avance.** Si tu docente lo solicita, escribe en la celda de configuración tu
> número de cuenta (o el alias que te asignen) y la clave del grupo. Así el equipo docente sabe
> en qué ejercicios necesita ayuda el grupo. Solo se envía el resultado de cada verificación,
> **nunca tu código**. Si no te lo piden, deja los campos vacíos.

> 💡 El verificador lee las variables del notebook. Si reinicias el entorno de ejecución, vuelve
> a ejecutar la celda de configuración y las celdas anteriores al ejercicio.

In [ ]:
# ⚙️ Configuración: ejecuta esta celda antes de empezar
import os
import sys
import urllib.request

REPOSITORIO = "https://raw.githubusercontent.com/ibonfilrivera/Curso_Python_FQ/main"

if os.path.isdir("../verificador"):      # Copia local del repositorio
    sys.path.insert(0, "..")
else:                                     # Google Colab: descarga el verificador
    os.makedirs("verificador", exist_ok=True)
    for archivo in ["__init__.py", "nucleo.py", "registro.py", "configuracion.py",
                    "sesion3.py"]:
        urllib.request.urlretrieve(f"{REPOSITORIO}/verificador/{archivo}",
                                   f"verificador/{archivo}")

from verificador.sesion3 import *

# 📋 Registro de avance (solo si tu docente lo pide): tu identificador y la clave del grupo
iniciar_registro(alumno="", clave="")

## **Importar bibliotecas**

Como vimos con NumPy en la sesión anterior, las bibliotecas se cargan con `import` y es
costumbre darles un alias corto:

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# **math, SymPy y SciPy**

El módulo `math` incluye funciones y constantes matemáticas básicas.

In [ ]:
print(math.sin(math.pi))       # Seno (el resultado es ~0 por redondeo numérico)
print(math.cos(math.pi / 3))   # Coseno
print(math.sqrt(100))          # Raíz cuadrada
print(math.log(math.e))        # Logaritmo natural
print(math.log10(100))         # Logaritmo base 10
print(math.log2(8))            # Logaritmo base 2

**SymPy** hace cálculo **simbólico**: manipula expresiones algebraicas en lugar de números.

In [ ]:
import sympy as sp

x = sp.symbols("x")
expresion = x**2 * sp.exp(-x)

print("Derivada:", sp.diff(expresion, x))
print("Integral:", sp.integrate(expresion, x))

**SciPy** reúne métodos numéricos, estadísticos y de optimización para el trabajo científico.
Por ejemplo, con `scipy.stats` podemos analizar una curva de calibración de absorbancia contra
concentración:

In [ ]:
from scipy.stats import describe, linregress

concentraciones = [0.0, 2.0, 4.0, 6.0, 8.0, 10.0]      # mg/L
absorbancias = [0.012, 0.225, 0.451, 0.645, 0.852, 1.058]

print(describe(absorbancias))                          # Estadística descriptiva

ajuste = linregress(concentraciones, absorbancias)     # Regresión lineal
print(f"Pendiente: {ajuste.slope:.4f} L/mg")
print(f"Ordenada:  {ajuste.intercept:.4f}")
print(f"r²:        {ajuste.rvalue**2:.5f}")

Con `curve_fit` de `scipy.optimize` podemos ajustar cualquier función, por ejemplo un polinomio de segundo grado:

In [ ]:
from scipy.optimize import curve_fit

def cuadratica(x, a, b, c):
    return a * x**2 + b * x + c

parametros, covarianza = curve_fit(cuadratica, concentraciones, absorbancias)
a, b, c = parametros
print(f"A = {a:.2e}·c² + {b:.4f}·c + {c:.4f}")

`scipy.optimize` también resuelve ecuaciones numéricamente (`fsolve`) y busca mínimos de
funciones (`minimize`):

In [ ]:
from scipy.optimize import fsolve, minimize

def polinomio(x):
    return (x + 2)**2 - 4

raiz = fsolve(polinomio, x0=-3)       # Busca una raíz cerca de x = -3
print(f"Raíz: x = {raiz[0]:.4f}")

minimo = minimize(polinomio, x0=3)    # Busca un mínimo empezando en x = 3
print(f"Mínimo en x = {minimo.x[0]:.4f}, f(x) = {minimo.fun:.4f}")

### **Ejercicio 1: Orden de reacción**

Se midió la concentración de un reactivo A a lo largo del tiempo. Las ecuaciones integradas de
velocidad son:

| Orden | Ecuación | Se grafica contra $t$ |
| :-: | :-: | :-: |
| 0 | $[A] = -kt + [A]_0$ | $[A]$ |
| 1 | $\ln[A] = -kt + \ln[A]_0$ | $\ln[A]$ |
| 2 | $\frac{1}{[A]} = kt + \frac{1}{[A]_0}$ | $\frac{1}{[A]}$ |

Haz las tres regresiones lineales con `linregress`, compara sus $r^2$ y guarda:
- `mejor_orden`: el orden (0, 1 o 2) que mejor describe los datos.
- `k`: la constante de velocidad (positiva) para ese orden.

In [ ]:
tiempo = [0, 10, 20, 30, 40, 50]                      # min
conc = [1.000, 0.607, 0.368, 0.223, 0.135, 0.082]     # mol/L

In [ ]:
ln_conc = [math.log(c) for c in conc]
inv_conc = [1 / c for c in conc]

ajustes = {
    0: linregress(tiempo, conc),
    1: linregress(tiempo, ln_conc),
    2: linregress(tiempo, inv_conc),
}
for orden, ajuste in ajustes.items():
    print(f"Orden {orden}: r² = {ajuste.rvalue**2:.5f}")

mejor_orden = max(ajustes, key=lambda o: ajustes[o].rvalue**2)
k = -ajustes[1].slope
print(f"Mejor ajuste: orden {mejor_orden}, k = {k:.4f} min⁻¹")

In [ ]:
# Verifica tu respuesta
ej1.verificar()

# **Matplotlib**

Matplotlib es la biblioteca de visualización más usada en ciencia. Aunque tiene varias formas
de uso, recomendamos la interfaz **orientada a objetos**:

```python
fig, ax = plt.subplots()      # fig: la figura completa; ax: los ejes donde se dibuja
ax.plot(x, y)                 # Dibujar
ax.set_xlabel("...")          # Personalizar
plt.show()                    # Mostrar
```

Una gráfica científica de calidad siempre tiene: **ejes etiquetados con unidades**, una
**leyenda** si hay más de una serie, y un tamaño de letra legible.

📎 Referencias: [hojas de referencia rápida](https://matplotlib.org/cheatsheets/) y
[galería de ejemplos](https://matplotlib.org/stable/gallery/index.html).

### **Gráfica de líneas: decaimiento radiactivo**

Retomemos el carbono-14 de la sesión anterior, ahora con $m(t) = m_0 \left(\frac{1}{2}\right)^{t/t_{1/2}}$.
Con NumPy generamos 200 tiempos de una sola vez con `np.linspace`.

In [ ]:
t_vida_media = 5730                              # años
t = np.linspace(0, 5 * t_vida_media, 200)        # 200 tiempos entre 0 y 5 vidas medias
masa = 1.0 * 0.5**(t / t_vida_media)             # g

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(t, masa, color="tab:blue", linewidth=2, label="¹⁴C")
ax.axhline(0.5, color="gray", linestyle="--", label="Mitad de la masa inicial")
ax.set_xlabel("Tiempo (años)")
ax.set_ylabel("Masa (g)")
ax.set_title("Decaimiento radiactivo del carbono-14")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

### **Varias gráficas en una figura**

`plt.subplots(filas, columnas)` crea una cuadrícula de ejes. Veamos los datos cinéticos del
Ejercicio 1 con las tres transformaciones: la que se vea como una recta indica el orden de reacción.

In [ ]:
tiempo_arr = np.array(tiempo)
conc_arr = np.array(conc)

transformaciones = [
    (conc_arr, "[A] (mol/L)", "Orden 0"),
    (np.log(conc_arr), "ln [A]", "Orden 1"),
    (1 / conc_arr, "1/[A] (L/mol)", "Orden 2"),
]

fig, ejes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, (y, etiqueta, titulo) in zip(ejes, transformaciones):
    ajuste_orden = linregress(tiempo_arr, y)
    ax.plot(tiempo_arr, y, "o", label="Datos")
    ax.plot(tiempo_arr, ajuste_orden.slope * tiempo_arr + ajuste_orden.intercept, "--",
            label=f"r² = {ajuste_orden.rvalue**2:.4f}")
    ax.set_xlabel("t (min)")
    ax.set_ylabel(etiqueta)
    ax.set_title(titulo)
    ax.legend()

fig.tight_layout()     # Evita que se encimen las etiquetas
plt.show()

### **Guardar una figura**

Para un reporte o una tesis, exporta con alta resolución (300 ppp) o en formato vectorial (PDF,
SVG). En Colab, el archivo aparece en el panel de archivos 📁 de la izquierda.

```python
fig.savefig("cinetica.png", dpi=300, bbox_inches="tight")
fig.savefig("cinetica.pdf", bbox_inches="tight")
```

### **Ejercicio 2: Curva de calibración**

Con los datos de `concentraciones` y `absorbancias` de la sección de SciPy:

1. Crea la figura `fig_calibracion` con los puntos experimentales (`ax.scatter`) y la recta del
   ajuste lineal (`ax.plot`).
2. Etiqueta ambos ejes con sus unidades y agrega una leyenda.
3. Una muestra problema tiene una absorbancia de 0.500. Calcula su concentración con la recta
   de calibración y guárdala en `conc_problema`.

> Aquí *suponemos* que la absorbancia es proporcional a la concentración. En el Ejercicio 3
> comprobaremos esa suposición con datos de un espectrofotómetro.

In [ ]:
ajuste = linregress(concentraciones, absorbancias)
x = np.array(concentraciones)

fig_calibracion, ax = plt.subplots(figsize=(6, 4))
ax.scatter(concentraciones, absorbancias, color="tab:blue", label="Datos experimentales")
ax.plot(x, ajuste.slope * x + ajuste.intercept, color="tab:red",
        label=f"A = {ajuste.slope:.4f}·c + {ajuste.intercept:.4f} (r² = {ajuste.rvalue**2:.4f})")
ax.set_xlabel("Concentración (mg/L)")
ax.set_ylabel("Absorbancia")
ax.set_title("Curva de calibración")
ax.legend()
plt.show()

conc_problema = (0.500 - ajuste.intercept) / ajuste.slope
print(f"Concentración de la muestra problema: {conc_problema:.2f} mg/L")

In [ ]:
# Verifica tu respuesta
ej2.verificar()

# **Pandas: datos del laboratorio**

Pandas es la biblioteca más usada para analizar datos tabulares. Su estructura principal es el
**DataFrame**, una tabla con filas y columnas con nombre, parecida a una hoja de cálculo. La
mayoría de los instrumentos (espectrofotómetros, potenciómetros, cromatógrafos) exportan sus
lecturas como CSV o Excel, y Pandas las lee con `pd.read_csv()` o `pd.read_excel()`.

Empezaremos con las lecturas de un espectrofotómetro: disoluciones estándar de KMnO₄ medidas
**por triplicado** a 525 nm en una celda de 1.00 cm, más una muestra problema. Los datos son
simulados, pero reproducen el comportamiento de un instrumento real.

In [ ]:
from pathlib import Path

def leer_datos(archivo):
    """Lee un CSV de la carpeta data/: la copia local si existe o la de GitHub (en Colab)."""
    ruta_local = Path("../data") / archivo
    return pd.read_csv(ruta_local if ruta_local.exists() else f"https://raw.githubusercontent.com/ibonfilrivera/Curso_Python_FQ/main/data/{archivo}")

datos_lb = leer_datos("lambert_beer_kmno4.csv")

print(f"{datos_lb.shape[0]} lecturas, columnas: {list(datos_lb.columns)}")
datos_lb.head(6)

Una columna se selecciona por su nombre, y las filas se **filtran** escribiendo una condición
entre corchetes. Varias condiciones se combinan con `&` (y) o `|` (o), cada una entre paréntesis:

In [ ]:
print("Absorbancia máxima:", datos_lb["absorbancia"].max())

# Solo las lecturas de la muestra problema
datos_lb[datos_lb["tipo"] == "problema"]

Para resumir datos por grupos usamos `groupby`, el equivalente a una *tabla dinámica* de Excel:
se agrupan las filas que comparten un valor y se aplica una función a cada grupo.

In [ ]:
# ¿Cuántas lecturas hay de cada tipo, y cuál es su absorbancia promedio?
datos_lb.groupby("tipo")["absorbancia"].agg(["count", "mean"])

### **Ejercicio 3: Determinación de la ley de Lambert-Beer**

La ley de Lambert-Beer relaciona la absorbancia con la concentración:

$$A = \varepsilon\, b\, c$$

donde $\varepsilon$ es la absortividad molar (L·mol⁻¹·cm⁻¹), $b$ el paso óptico (cm) y $c$ la
concentración (mol/L). Con las lecturas de `datos_lb` vamos a comprobar si se cumple, en qué
intervalo, y a usarla para cuantificar la muestra problema.

**3a. Promedio de réplicas.** Filtra los estándares (`tipo == "estandar"`) y agrúpalos por
concentración para obtener el DataFrame `resumen`, con las columnas `promedio` y `desviacion`
(desviación estándar) de la absorbancia.

In [ ]:
estandares = datos_lb[datos_lb["tipo"] == "estandar"]
resumen = estandares.groupby("concentracion_mol_L")["absorbancia"].agg(
    promedio="mean", desviacion="std")
resumen

In [ ]:
# Verifica tu respuesta
ej3a.verificar()

**3b. Absortividad molar.** Los espectrofotómetros pierden linealidad a absorbancias altas (por
la luz parásita, entre otras causas), así que la ley solo se cumple en un intervalo.

1. Ajusta una recta con **todos** los estándares y guarda su r² en `r2_todos`.
2. Ajusta otra solo con los estándares cuya absorbancia promedio sea **≤ 1.0** (DataFrame
   `lineal`, ajuste `ajuste_lb`) y guarda su r² en `r2_lineal`.
3. Calcula la absortividad molar `epsilon` a partir de la pendiente (b = 1.00 cm).
4. Grafica en `fig_lb` los promedios con barras de error y la recta del intervalo lineal.

In [ ]:
b = 1.00   # cm

ajuste_todos = linregress(resumen.index, resumen["promedio"])
r2_todos = ajuste_todos.rvalue**2

lineal = resumen[resumen["promedio"] <= 1.0]
ajuste_lb = linregress(lineal.index, lineal["promedio"])
r2_lineal = ajuste_lb.rvalue**2
epsilon = ajuste_lb.slope / b

fig_lb, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(resumen.index * 1e3, resumen["promedio"], yerr=resumen["desviacion"],
            fmt="o", capsize=3, label="Estándares (promedio ± s)")
c = np.linspace(0, lineal.index.max(), 50)
ax.plot(c * 1e3, ajuste_lb.slope * c + ajuste_lb.intercept, color="tab:red",
        label=f"Ajuste en A ≤ 1: ε = {epsilon:.0f} L/(mol·cm)")
c_extra = np.linspace(lineal.index.max(), resumen.index.max(), 20)
ax.plot(c_extra * 1e3, ajuste_lb.slope * c_extra + ajuste_lb.intercept, color="tab:red",
        linestyle="--", alpha=0.5, label="Extrapolación de la recta")
ax.axhline(1.0, color="gray", linestyle=":", label="Límite del intervalo lineal")
ax.set_xlabel("Concentración de KMnO₄ (mmol/L)")
ax.set_ylabel("Absorbancia a 525 nm")
ax.legend()
plt.show()

print(f"Todos los puntos: r² = {r2_todos:.4f}")
print(f"Intervalo lineal: r² = {r2_lineal:.5f}, ε = {epsilon:.0f} L/(mol·cm)")

In [ ]:
# Verifica tu respuesta
ej3b.verificar()

**3c. Muestra problema.** Promedia las tres lecturas de la muestra problema, calcula su
concentración con `ajuste_lb` (`conc_problema_lb`, en mol/L) y guarda en `dentro_intervalo` si
esa concentración queda dentro del intervalo de los estándares lineales (`True` o `False`).

> 🤔 **Para reflexionar:** si otra muestra diera A = 1.8, ¿por qué convendría diluirla antes de
> medirla, en lugar de extrapolar la recta?

In [ ]:
problema = datos_lb[datos_lb["tipo"] == "problema"]
a_problema = problema["absorbancia"].mean()

conc_problema_lb = (a_problema - ajuste_lb.intercept) / ajuste_lb.slope
dentro_intervalo = lineal.index.min() <= conc_problema_lb <= lineal.index.max()

print(f"A = {a_problema:.3f} → c = {conc_problema_lb:.3e} mol/L")
print(f"¿Dentro del intervalo calibrado? {dentro_intervalo}")

In [ ]:
# Verifica tu respuesta
ej3c.verificar()

## **Bases de datos grandes: AqSolDB**

Las mismas herramientas sirven para tablas mucho más grandes. Usaremos **AqSolDB**, una base de
datos curada con la solubilidad acuosa de 9 982 compuestos
([Sorkun *et al.*, *Scientific Data* **6**, 143 (2019)](https://doi.org/10.1038/s41597-019-0151-1)).
Algunas de sus columnas son:

| Columna | Significado |
| :--- | :--- |
| `Name`, `SMILES` | Nombre y estructura del compuesto |
| `Solubility` | log S, con S la solubilidad en mol/L |
| `MolWt` | Masa molar (g/mol) |
| `MolLogP` | Coeficiente de partición octanol/agua (log P) |
| `NumHDonors`, `NumHAcceptors` | Donadores y aceptores de puentes de hidrógeno |
| `TPSA` | Área superficial polar (Å²) |

In [ ]:
df = leer_datos("curated_solubility.csv")

print(f"La tabla tiene {df.shape[0]} filas y {df.shape[1]} columnas")
df.head()

Hagamos un análisis preliminar. `describe()` resume las columnas numéricas:

In [ ]:
df[["Solubility", "MolWt", "MolLogP", "NumHDonors", "NumHAcceptors"]].describe()

In [ ]:
# Compuestos con masa molar menor a 50 g/mol
df[df["MolWt"] < 50][["Name", "MolWt", "Solubility"]]

In [ ]:
# Los 5 compuestos más solubles
df.sort_values("Solubility", ascending=False)[["Name", "Solubility"]].head()

Pandas se integra con Matplotlib. Por ejemplo, un histograma de la solubilidad:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df["Solubility"], bins=50, color="tab:green", edgecolor="white")
ax.set_xlabel("log S (S en mol/L)")
ax.set_ylabel("Número de compuestos")
ax.set_title("Distribución de la solubilidad acuosa en AqSolDB")
plt.show()

### **Ejercicio 4: Regla de los 5 de Lipinski**

En el diseño de fármacos, la regla de Lipinski estima si un compuesto podría ser activo por vía
oral. Un buen candidato cumple **todas** estas condiciones:

- Masa molar ≤ 500 g/mol.
- log P ≤ 5.
- Donadores de puentes de hidrógeno ≤ 5.
- Aceptores de puentes de hidrógeno ≤ 10.

Guarda en `lipinski` las filas de `df` que cumplen la regla y en `porcentaje_lipinski` el
porcentaje de compuestos que la cumplen.

In [ ]:
lipinski = df[(df["MolWt"] <= 500)
              & (df["MolLogP"] <= 5)
              & (df["NumHDonors"] <= 5)
              & (df["NumHAcceptors"] <= 10)]

porcentaje_lipinski = 100 * len(lipinski) / len(df)
print(f"{len(lipinski)} moléculas ({porcentaje_lipinski:.1f} %) cumplen la regla de Lipinski")

In [ ]:
# Verifica tu respuesta
ej4.verificar()

### **Ejercicio 5 (integrador): lipofilicidad y solubilidad**

¿Los compuestos más lipofílicos son menos solubles en agua?

1. Crea la figura `fig_logp` con un diagrama de dispersión de `MolLogP` (eje x) contra
   `Solubility` (eje y), con ambos ejes etiquetados.
2. Calcula el coeficiente de correlación de Pearson entre ambas columnas y guárdalo en `r_logp`.
3. Interpreta: ¿qué signo tiene la correlación y qué significa químicamente?

In [ ]:
r_logp = df["MolLogP"].corr(df["Solubility"])

fig_logp, ax = plt.subplots(figsize=(6, 4))
ax.scatter(df["MolLogP"], df["Solubility"], s=5, alpha=0.3)
ax.set_xlabel("logP (octanol/agua)")
ax.set_ylabel("log S (mol/L)")
ax.set_title(f"Solubilidad acuosa contra lipofilicidad (r = {r_logp:.2f})")
plt.show()

In [ ]:
# Verifica tu respuesta
ej5.verificar()

# **RDKit: química computacional**

RDKit es una biblioteca de quimioinformática: interpreta estructuras moleculares (por ejemplo,
en notación **SMILES**), las dibuja y calcula propiedades. Como no viene instalada en Colab, la
instalamos primero.

In [ ]:
try:
    import rdkit
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rdkit"], check=True)

from rdkit import Chem, RDLogger
from rdkit.Chem import Draw

RDLogger.DisableLog("rdApp.*")   # Oculta advertencias de estructuras problemáticas

In [ ]:
aspirina = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O")
aspirina

Convertimos todos los SMILES de la tabla en objetos de RDKit (tarda unos segundos):

In [ ]:
df["Mols"] = df["SMILES"].apply(Chem.MolFromSmiles)

no_validas = df["Mols"].isna().sum()
print(f"Moléculas que RDKit no pudo interpretar: {no_validas}")

Draw.MolsToGridImage(df["Mols"][:8].tolist(), molsPerRow=4, subImgSize=(200, 200),
                     legends=[nombre[:25] for nombre in df["Name"][:8]])

### **Ejercicio 6: Búsqueda de subestructuras**

Con un patrón **SMARTS** podemos buscar fragmentos dentro de las moléculas. El anillo bencénico
aromático se escribe `"c1ccccc1"`.

1. Completa la función `tiene_benceno(mol)`, que devuelva `True` si la molécula contiene un
   anillo bencénico (y `False` si `mol` es `None`).
2. Aplícala para crear la columna `df["tiene_benceno"]`.
3. Guarda en `n_benceno` cuántos compuestos contienen benceno.

In [ ]:
patron_benceno = Chem.MolFromSmarts("c1ccccc1")

def tiene_benceno(mol):
    if mol is None:          # SMILES que RDKit no pudo interpretar
        return False
    return mol.HasSubstructMatch(patron_benceno)

df["tiene_benceno"] = df["Mols"].apply(tiene_benceno)
n_benceno = int(df["tiene_benceno"].sum())
print(f"{n_benceno} de {len(df)} moléculas contienen un anillo bencénico")

In [ ]:
# Verifica tu respuesta
ej6.verificar()

## **Resumen de la sesión**

| Biblioteca | Para qué sirve | Funciones clave |
| :--- | :--- | :--- |
| `math` | Funciones matemáticas básicas | `math.log`, `math.sqrt`, `math.pi` |
| SymPy | Cálculo simbólico | `sp.symbols`, `sp.diff`, `sp.integrate` |
| SciPy | Estadística, ajustes y ecuaciones | `linregress`, `curve_fit`, `fsolve` |
| Matplotlib | Gráficas | `plt.subplots`, `ax.plot`, `ax.scatter`, `ax.errorbar`, `fig.savefig` |
| Pandas | Datos tabulares | `pd.read_csv`, `df[condición]`, `groupby().agg()`, `describe` |
| RDKit | Quimioinformática | `Chem.MolFromSmiles`, `HasSubstructMatch` |

**Para seguir aprendiendo:**
- [Kaggle Learn: Pandas](https://www.kaggle.com/learn/pandas) y [Data Visualization](https://www.kaggle.com/learn/data-visualization).
- [Tutorial de introducción de RDKit](https://www.rdkit.org/docs/GettingStartedInPython.html).

## **Tu progreso**

Ejecuta la siguiente celda para ver cuántos ejercicios resolviste en esta sesión.

In [ ]:
progreso()